## Section 1: Transformer (Mini-BERT) (30 points)

# Assignment: Implementing and Understanding BERT (Mini-BERT)

**What you will do:**
1. Implement core components of the BERT (Mini-BERT) **from scratch in PyTorch**.
2. Pretrain the model with **Masked Language Modeling (MLM)** on **WikiText-2**.
3. Fine-tune the pretrained model for **binary sentiment classification** on **SST-2**.
4. Each **TODO** section has **raise NotImplementedError**. Please remove it after you have implemented the respective section.

**Constraints:**
- Make sure the runtime is set to GPU.
- Assignment uses a small configuration for the BERT:
  - `num_layers=4`, `d_model=256`, `num_heads=4`, `max_len=64`

**Disclaimer:**
- The results may not be great as we are pretraining and finetuning the model on a fraction of the datasets and fewer epochs, for reducing the training time, and focusing on understanding the fundamentals.


## Environment Setup

We install required libraries and import modules.


In [1]:
!pip -q install datasets transformers tqdm

In [3]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


## Tokenizer

We use the pretrained **BERT tokenizer** (`bert-base-uncased`).

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size
mask_token_id = tokenizer.mask_token_id
pad_token_id = tokenizer.pad_token_id
cls_token_id = tokenizer.cls_token_id
sep_token_id = tokenizer.sep_token_id

print("vocab_size:", vocab_size)
print("special token ids:", {"[PAD]": pad_token_id, "[CLS]": cls_token_id, "[SEP]": sep_token_id, "[MASK]": mask_token_id})


vocab_size: 30522
special token ids: {'[PAD]': 0, '[CLS]': 101, '[SEP]': 102, '[MASK]': 103}


## Datasets

## Pretraining corpus: WikiText-2 (raw text)
We'll use `wikitext-2-raw-v1` and pretrain using **MLM**.

## Fine-tuning task: SST-2
Binary sentiment classification (positive/negative).

In [5]:
# Loading the datasets
# Pretraining: WikiText-2 raw
wikitext = load_dataset("wikitext", "wikitext-2-raw-v1")
# Fine-tuning: SST-2
sst2 = load_dataset("glue", "sst2")

print("WikiText splits:", wikitext)
print("SST-2 splits:", sst2)


WikiText splits: DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
SST-2 splits: DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


## Quick peek at the data

In [6]:
print("WikiText example:")
print(wikitext["train"][3]["text"][:50])
print("\nSST-2 example:")
print(sst2["train"][0])

WikiText example:
 Senjō no Valkyria 3 : Unrecorded Chronicles ( Jap

SST-2 example:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


## Masked Language Modeling (MLM) Dataset

We implement the MLM corruption scheme:
- Select **15%** of *non-special* tokens for prediction.
- For selected tokens:
  - **80%** replace with `[MASK]`
  - **10%** replace with a random token id
  - **10%** keep unchanged
- `labels` should be:
  - original token id for selected positions
  - `-100` for all other positions (so loss ignores them)

### TODO: `mask_tokens`
**Expected outputs**
- `masked_input_ids`: `torch.LongTensor` of shape `(max_len,)`
- `labels`: `torch.LongTensor` of shape `(max_len,)`
- `labels` values are `-100` except at masked positions.


In [7]:
class MLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=64):
        self.texts = [t for t in texts if isinstance(t, str) and len(t.strip()) > 0]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def mask_tokens(self, input_ids):
        '''
        TODO:
        Implement 15% token masking with 80/10/10 rule.

        Inputs:
          input_ids: LongTensor (max_len,)

        Returns:
          masked_input_ids: LongTensor (max_len,)
          labels: LongTensor (max_len,) with -100 for non-predicted positions

        Expected:
          - Do NOT mask [CLS], [SEP], [PAD]
          - Keep shape exactly (max_len,)
        '''
        ######SHREYA CODE#######
        labels = input_ids.clone()
        #probability matrix for masking 15%
        probability_matrix = torch.full(labels.shape, 0.15)
        
        #prevent masking of special tokens like CLS, SEP, PAD: find them and set prob to 0
        special_tokens_mask = [
                self.tokenizer.get_special_tokens_mask(val, already_has_special = TRUE)
                for val in labels.tolist()
        ]

        #input_ids is (max_len) so no need for comprehension if 1d. 
        special_tokens_mask = torch.tensor(special_ttokens_mask, dtype = torch.bool)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)

        #which tokens are we masking
        masked_indices = torch.bernoulli(probability_matrix).bool()

        #set labels for non-masked positions to -100
        labels[~masked_indices] = -100

        #apply the 80/10/10
        masked_input_ids = input_ids.clone()

        #80% of the time replace with mask
        indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
        masked_input_ids[indices_replaced] = self.tokenizer.convert_tokens_to_ids(self.tokenizer.mask_token)

        #10% of the time of remaining 20%, replace with random token
        indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(len(self.tokenizer),labels.shape, dtype=torch.long)
        masked_input_ids[indices_random]  = random_words[indices_random]
        return masked_input_ids, labels
      
        #https://medium.com/@ngiengkianyew/implementation-of-a-simple-masked-language-model-6a3bc18912c8 : source

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)  # (max_len,)
        masked_input_ids, labels = self.mask_tokens(input_ids)
        return masked_input_ids, labels


# Build a small MLM dataset
MLM_TRAINSET_SIZE = 10000
MAX_LEN = 64
wikitext_train_texts = wikitext["train"]["text"]
mlm_train_ds = MLMDataset(
    wikitext_train_texts[:MLM_TRAINSET_SIZE],
    tokenizer,
    max_len=MAX_LEN)
mlm_train_loader = DataLoader(
    mlm_train_ds,
    batch_size=64,
    num_workers=2,
    pin_memory=True,
    shuffle=True)

print("MLM dataset size:", len(mlm_train_ds))


MLM dataset size: 6520


## Sanity checks

In [ ]:
def show_mlm_example(ds, n=3):
    for i in range(n):
        x, y = ds[random.randrange(len(ds))]
        # show masked vs labels
        masked_tokens = tokenizer.convert_ids_to_tokens(x.tolist())
        label_tokens = ["_" if t == -100 else tokenizer.convert_ids_to_tokens([t])[0] for t in y.tolist()]
        print("masked:", masked_tokens[:30])
        print("label :", label_tokens[:30])
        print("---")

show_mlm_example(mlm_train_ds, n=2)

# Shape checks
x, y = mlm_train_ds[0]
print(x.shape, y.shape)
assert x.shape == (MAX_LEN,) and y.shape == (MAX_LEN,)


## Implement Mini-BERT Encoder

We implement a small BERT-style encoder:
- Token embeddings + positional embeddings
- Stack of Transformer encoder layers
- LayerNorm

### Configuration (Colab-safe)
- `d_model=256`
- `num_heads=4`  ⇒ per-head dim `d_k = 64`
- `num_layers=4`
- `max_len=64`

### 5.1 Multi-Head Self-Attention (TODO)

**Expected shapes**
- Input `x`: `(batch, seq_len, d_model)`
- `Q, K, V`: `(batch, num_heads, seq_len, d_k)`
- Attention scores: `(batch, num_heads, seq_len, seq_len)`
- Output: `(batch, seq_len, d_model)`

**Masking**
- We use an attention mask where **1 means keep**, **0 means pad**.
- If mask is provided as `(batch, seq_len)`, expand to broadcast over heads and query positions.
- Before softmax: set masked positions to a very negative value (e.g., `-1e9`).


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        '''
        TODO:
        Implement multi-head self-attention.

        Inputs:
          x: (B, T, D)
          attn_mask: optional (B, T) with 1 for tokens, 0 for pads

        Returns:
          out: (B, T, D)

        Expected:
          - Use scaled dot-product: softmax(QK^T / sqrt(d_k)) V
          - Apply mask to scores before softmax (pads should not be attended to)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement MultiHeadSelfAttention")
        # ============================


## Feed-Forward Network (TODO)

BERT uses a 2-layer MLP with GELU:
`FFN(x) = Linear2(GELU(Linear1(x)))`

**Expected shapes**
- Input: `(B, T, D)`
- Output: `(B, T, D)`


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model=256, hidden_dim=1024, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        '''
        TODO:
        Implement Linear -> GELU -> Dropout -> Linear

        Expected:
          x: (B, T, D)  ->  out: (B, T, D)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement FeedForward")
        # ============================


## Transformer Encoder Layer (TODO)

A standard encoder layer is:
1. Self-attention + residual + LayerNorm
2. Feed-forward + residual + LayerNorm

**Expected shapes**
- Input: `(B, T, D)`
- Output: `(B, T, D)`

**Important**
- Use dropout on the sublayer outputs before adding residual.


In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, 4 * d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        '''
        TODO:
        Implement:
          a = self.attn(x, attn_mask)
          x = norm1(x + dropout(a))
          f = self.ff(x)
          x = norm2(x + dropout(f))

        Expected:
          x: (B, T, D)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement EncoderLayer")
        # ============================


## Mini-BERT Encoder (TODO)

We build the encoder:
- Token embedding: `nn.Embedding(vocab_size, d_model)`
- Positional embedding: `nn.Embedding(max_len, d_model)`
- `num_layers` encoder blocks
- Final LayerNorm

**Expected shapes**
- Input `input_ids`: `(B, T)`
- Output `hidden_states`: `(B, T, D)`


In [ ]:
class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_layers=4, num_heads=4, max_len=64, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attn_mask=None):
        '''
        TODO:
        1) Create positions: (B, T) with values 0..T-1
        2) Embed tokens + positions, apply dropout
        3) Pass through encoder layers
        4) Apply final LayerNorm

        Expected:
          input_ids: (B, T)
          return: (B, T, D)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement MiniBERT")
        # ============================


## Shape unit tests (should pass after TODOs are done)

In [ ]:
# After implementing attention / encoder / MiniBERT, run this cell.

B, T, D = 2, MAX_LEN, 256
dummy_ids = torch.randint(low=0, high=vocab_size, size=(B, T))
dummy_mask = (dummy_ids != pad_token_id).long()

model = MiniBERT(vocab_size=vocab_size, d_model=256, num_layers=2, num_heads=4, max_len=MAX_LEN).to(device)
out = model(dummy_ids.to(device), dummy_mask.to(device))
print("out shape:", out.shape)
assert out.shape == (B, T, 256)


## MLM Head + Pretraining Loop

To pretrain, we add a vocabulary projection head on top of MiniBERT:
- Input: `(B, T, D)`
- Output logits: `(B, T, vocab_size)`

Loss:
- CrossEntropy over vocabulary for masked positions only (labels use `-100` elsewhere).

### TODO: implement forward in `MaskedLMHead` (simple)
**Expected output**
- `logits.shape == (B, T, vocab_size)`


In [ ]:
class MaskedLMHead(nn.Module):
    def __init__(self, bert: MiniBERT, vocab_size, d_model=256):
        super().__init__()
        self.bert = bert
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, attn_mask=None):
        '''
        TODO:
        1) Get hidden states from bert: (B, T, D)
        2) Project to vocab logits: (B, T, V)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement MaskedLMHead")
        # ============================


### Pretraining loop (TODO blocks inside)

We keep pretraining short (e.g., 1 epoch on a subset) due to Colab constraints.

**Expected**
- Code runs without shape errors.
- Loss is a finite scalar and should generally decrease.


In [ ]:
def pretrain_mlm(model, dataloader, epochs=1, lr=3e-4):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for epoch in range(epochs):
        total_loss = 0.0
        for input_ids, labels in tqdm(dataloader, desc=f"MLM pretrain epoch {epoch+1}"):
            input_ids = input_ids.to(device)          # (B, T)
            labels = labels.to(device)                # (B, T)
            attn_mask = (input_ids != pad_token_id).long()  # (B, T)

            # TODO:
            # 1) Forward -> logits (B, T, V)
            # 2) Compute loss. Hint: reshape logits to (B*T, V) and labels to (B*T)
            # 3) Backprop + optimizer step
            # 4) Accumulate loss
            # ====== YOUR CODE HERE ======
            raise NotImplementedError("TODO: implement pretrain_mlm")
            # ============================

        avg = total_loss / max(1, len(dataloader))
        print(f"Epoch {epoch+1}: avg_loss={avg:.4f}")


bert = MiniBERT(vocab_size=vocab_size, d_model=256, num_layers=4, num_heads=4, max_len=MAX_LEN).to(device)
mlm_model = MaskedLMHead(bert, vocab_size=vocab_size, d_model=256).to(device)
pretrain_mlm(mlm_model, mlm_train_loader, epochs=1, lr=3e-4)


## Fine-tuning on SST-2

We fine-tune by adding a classification head over the `[CLS]` representation:
- Input IDs: `(B, T)`
- Encoder output: `(B, T, D)`
- `[CLS]` vector: `hidden[:, 0, :]`  → shape `(B, D)`
- Logits: `(B, 2)`

### TODO: SST-2 Dataset + Classifier forward + training step
**Expected outputs**
- Classifier logits: `(B, 2)`
- Loss: scalar
- Accuracy improves above random chance.


In [ ]:
class SST2Dataset(Dataset):
    def __init__(self, split, tokenizer, max_len=64):
        self.data = split
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sent = self.data[idx]["sentence"]
        label = self.data[idx]["label"]
        enc = self.tokenizer(
            sent,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)
        return input_ids, attn_mask, torch.tensor(label, dtype=torch.long)


# Dataset size
FINE_TUNE_SIZE = 10000

sst_train_ds = SST2Dataset(
    sst2["train"].select(range(FINE_TUNE_SIZE)),
    tokenizer,
    max_len=MAX_LEN)
sst_val_ds = SST2Dataset(
    sst2["validation"],
    tokenizer,
    max_len=MAX_LEN)

sst_train_loader = DataLoader(
    sst_train_ds,
    batch_size=64,
    num_workers=2,
    pin_memory=True,
    shuffle=True)
sst_val_loader = DataLoader(
    sst_val_ds,
    batch_size=64,
    shuffle=False)

print("SST-2 train size:", len(sst_train_ds))
print("SST-2 val size:", len(sst_val_ds))


In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, bert: MiniBERT, d_model=256, num_classes=2):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attn_mask=None):
        '''
        TODO:
        1) Get hidden states: (B, T, D)
        2) Extract CLS: hidden[:, 0, :] -> (B, D)
        3) Project to logits: (B, 2)
        '''
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement SentimentClassifier")
        # ============================


In [ ]:
@torch.no_grad()
def eval_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    for input_ids, attn_mask, labels in dataloader:
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        labels = labels.to(device)
        logits = model(input_ids, attn_mask)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
    return correct / max(1, total)


def finetune(model, train_loader, val_loader, epochs=2, lr=2e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for input_ids, attn_mask, labels in tqdm(train_loader, desc=f"finetune epoch {epoch+1}"):
            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)
            labels = labels.to(device)

            # TODO:
            # 1) Forward -> logits (B, 2)
            # 2) loss = CE(logits, labels)
            # 3) Backprop + step
            # 4) Accumulate loss
            # ====== YOUR CODE HERE ======
            raise NotImplementedError("TODO: implement finetune training step")
            # ============================

        val_acc = eval_accuracy(model, val_loader)
        avg_loss = total_loss / max(1, len(train_loader))
        print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, val_acc={val_acc:.4f}")


classifier = SentimentClassifier(bert, d_model=256).to(device)
finetune(classifier, sst_train_loader, sst_val_loader, epochs=2, lr=2e-4)


## Section 2: Vision Transformer (ViT) Architecture (20 Points)

In this section, we implement a simplified Vision Transformer for image classification on MNIST.
The model consists of three main components:

1. Patch Embedding: Converts the input image into a sequence of flattened patches and projects them into an embedding space, along with learnable positional encodings.
2. Transformer Encoder Blocks: A stack of self-attention and feed-forward layers with residual connections and layer normalization.
3. Classification Head: Uses the learned representation to predict the final class label.


# Setup

As usual, we start with basic data loading and preprocessing.


In [ ]:
!pip install einops

In [ ]:
!pip install torchinfo

In [ ]:
#Import necessary libraries
import torch
from torch import nn
from torch import nn, einsum
import torch.nn.functional as F
from torch import optim

from einops import rearrange, repeat
from einops.layers.torch import Rearrange
import numpy as np
import torchvision
import time
from torchinfo import summary
from torch.utils.data import DataLoader, Subset

# Load the data

In [ ]:
torch.manual_seed(42)

DOWNLOAD_PATH = '/data/mnist'
BATCH_SIZE_TRAIN = 100
BATCH_SIZE_TEST = 1000
train_size = 5000
test_size = 1000

transform_mnist = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize((0.1307,), (0.3081,))])

train_set = torchvision.datasets.MNIST(DOWNLOAD_PATH, train=True, download=True,
                                       transform=transform_mnist)

test_set = torchvision.datasets.MNIST(DOWNLOAD_PATH, train=False, download=True,
                                      transform=transform_mnist)

train_indices = torch.randperm(len(train_set))[:train_size]
test_indices = torch.randperm(len(test_set))[:test_size]

train_subset = Subset(train_set, train_indices)
test_subset = Subset(test_set, test_indices)

# Create loaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE_TEST, shuffle=False)

print("Train size:", len(train_subset))
print("Test size:", len(test_subset))


In [ ]:
import matplotlib.pyplot as plt

# Function to show a batch of images
def show_batch(data_loader):
    batch = next(iter(data_loader))
    images, labels = batch
    grid = torchvision.utils.make_grid(images, nrow=10)
    plt.figure(figsize=(12, 12))
    plt.imshow(grid.numpy().transpose((1, 2, 0)))
    plt.title("Batch of Training Images")
    plt.axis('off')
    plt.show()

# Print images for validation
show_batch(train_loader)

In [ ]:
def pair(t):
    return t if isinstance(t, tuple) else (t, t)

##Task: Implement the FeedForward Network

Complete the `forward()` method of the `FeedForward` class.

The input `x` has shape:

(B, T, D)

Where:
- **B** = batch size  
- **T** = number of tokens  
- **D** = embedding dimension  

### Follow These Steps:

1. Apply the first linear layer `self.fc1`.
2. Apply a **ReLU activation**.
3. Apply `self.dropout`.
4. Apply the second linear layer `self.fc2`.
5. Apply dropout again.

The output shape must remain (B, T, D).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, dim)
    def forward(self, x):
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement FeedForward")
        # ============================

##Task: Implement Multi-Head Self-Attention (Forward Pass)

Complete the `forward()` method of the `Attention` class.

The input tensor `x` has shape:

(B, T, D)

Where:
- **B** = batch size  
- **T** = number of tokens (patches + CLS token)  
- **D** = embedding dimension  

### Follow these steps in order:

1. Compute **Q, K, V** using the linear layers.  
2. Reshape to split into multiple heads → (B, heads, T, d_k).  
3. Compute scaled dot-product attention: QKᵀ / √d_k.  
4. Apply **softmax** over the last dimension.  
5. Multiply attention weights with **V**.  
6. Concatenate heads back to shape (B, T, D).  
7. Apply the final output projection.

Maintain correct tensor shapes at every step.

In [ ]:
import math
import torch
from torch import nn

class Attention(nn.Module):
    """
    This class implments the Multi-Head Self Attention
    """
    def __init__(self, dim, heads=4, dim_head=None, dropout=0.):
        super().__init__()

        assert dim % heads == 0, "Embedding dimension must be divisible by number of heads"

        self.dim = dim
        self.heads = heads
        self.d_k = dim // heads if dim_head is None else dim_head

        # If dim_head is provided, total projection dimension changes
        inner_dim = self.d_k * heads

        # Linear projections for Q, K, V (ViT style)
        self.q_linear = nn.Linear(dim, inner_dim, bias=False)
        self.k_linear = nn.Linear(dim, inner_dim, bias=False)
        self.v_linear = nn.Linear(dim, inner_dim, bias=False)

        # Output projection back to original embedding dimension
        self.out = nn.Linear(inner_dim, dim)

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Scaling factor for stability
        self.scale = math.sqrt(self.d_k)

    def forward(self, x):
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement Attention")
        # ============================

##Task: Implement the Forward Pass of One Transformer Encoder Block

Complete the `forward()` method of the `TransformerEncoder` class.

The input `x` has shape:

(B, T, D)

Where:
- **B** = batch size  
- **T** = number of tokens  
- **D** = embedding dimension  

### Follow These Steps in Order:

1. **LayerNorm + Attention**
   - Apply `self.norm1` to `x`.
   - Pass the result through `self.attn`.
   - Add the original input (`x`) using a residual connection.

2. **LayerNorm + FeedForward**
   - Apply `self.norm2` to the updated `x`.
   - Pass the result through `self.ff`.
   - Add the residual connection again.

Return the final output.

Each sub-layer must preserve the shape (B, T, D).

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()

        # ---- Attention Sub-layer ----
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)

        # ---- FeedForward Sub-layer ----
        self.norm2 = nn.LayerNorm(dim)
        self.ff = FeedForward(dim, mlp_dim, dropout=dropout)

    def forward(self, x):
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement TranformerEncoder")
        # ==========================

##Task: Implement the Forward Pass of the Transformer

Complete the `forward()` method of the `Transformer` class.

The input `x` has shape:

(B, T, D)

Where:
- **B** = batch size  
- **T** = number of tokens  
- **D** = embedding dimension  

### What You Need to Do:

1. Iterate through each encoder block stored in `self.layers`.
2. Pass `x` sequentially through every `TransformerEncoder`.
3. Update `x` after each block.
4. Return the final output after all layers are applied.

Conceptually:

Input → Encoder₁ → Encoder₂ → ... → Encoderₙ → Output

Do not change tensor shapes. Each encoder block preserves shape (B, T, D).

In [ ]:
class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()

        self.layers = nn.ModuleList()

        for _ in range(depth):
            self.layers.append(
                TransformerEncoder(dim, heads, dim_head, mlp_dim, dropout)
            )

    def forward(self, x):
        # ====== YOUR CODE HERE ======
        raise NotImplementedError("TODO: implement Transformer")
        # ============================

##Task: Implement the Forward Pass of Vision Transformer

Complete the `forward()` method of the `ViT` class.

The input `img` has shape:

(B, C, H, W)

Follow the steps below **in order**:

1. **Patch Embedding**  
   Convert the image into patch tokens using `self.to_patch_embedding(img)`.

2. **Add CLS Token**  
   - Expand `self.cls_token` to match the batch size.  
   - Concatenate it to the beginning of the patch tokens.

3. **Add Positional Embeddings**  
   Add `self.pos_embedding` to the token sequence.

4. **Apply Dropout**  
   Pass the tokens through `self.dropout`.

5. **Transformer Encoder**  
   Pass the sequence through `self.transformer`.

6. **Pooling**  
   - If `pool == 'cls'`, select the first token.  
   - If `pool == 'mean'`, average across the token dimension.

7. **Classification Head**  
   Pass the final representation through `self.mlp_head`.

Ensure tensor shapes remain consistent throughout the pipeline.

In [ ]:
class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth, heads, mlp_dim, pool = 'cls', channels = 3, dim_head = 64, dropout = 0., emb_dropout = 0.):
        super().__init__()
        # Image & Patch Sizes
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)

        # Validate Patch Compatibility
        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'

        # Compute Patch Info
        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width
        assert pool in {'cls', 'mean'}, 'pool type must be either cls (cls token) or mean (mean pooling)'

        # Patch Embedding Layer (Rearrange + Linear Projection)
        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1 = patch_height, p2 = patch_width),
            nn.Linear(patch_dim, dim),
        )

        # Positional Embedding & CLS Token
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.dropout = nn.Dropout(emb_dropout)

        # Transformer Encoder
        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        # Pooling strategy chosen during initialization.
        self.pool = pool

        # Classification Head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img):
        # ====== YOUR CODE HERE ======
        # Convert Image to Patch Tokens

        # Add CLS Token


        # Add Positional Embeddings


        # Dropout Layer


        # Pass to the Transformer Encoder


        # Pooling


        # Classification Head

        raise NotImplementedError("TODO: implement ViT")
        # ============================

In [ ]:
model = ViT(
    image_size=28,
    patch_size=4,
    num_classes=10,
    channels=1,
    dim=64,
    depth=6,
    heads=4,
    mlp_dim=128
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.003)

Let's see how the model looks like.

In [ ]:
summary(model)

This is it, 4 transformer blocks, followed by a linear classification layer. Let us quickly see how many trainable parameters are present in this model.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(count_parameters(model))

About half a million. Not too bad; the bigger NLP type models have several tens of millions of parameters. But since we are training on MNIST this should be more than sufficient.

# Training and Testing

In [ ]:
def train_epoch(model, optimizer, data_loader, loss_history):
    total_samples = len(data_loader.dataset)
    model.train()

    for i, (data, target) in enumerate(data_loader):
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad()
        output = F.log_softmax(model(data), dim=1)
        loss = F.nll_loss(output, target)

        loss.backward()
        optimizer.step()

        if i % 100 == 0:
            print(f"[{i * len(data):5}/{total_samples:5}] "
                  f"({100 * i / len(data_loader):3.0f}%) "
                  f"Loss: {loss.item():6.4f}")
            loss_history.append(loss.item())


In [ ]:
def evaluate(model, data_loader, loss_history):
    model.eval()

    total_samples = len(data_loader.dataset)
    correct_samples = 0
    total_loss = 0

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)

            output = F.log_softmax(model(data), dim=1)
            loss = F.nll_loss(output, target, reduction='sum')

            _, pred = torch.max(output, dim=1)

            total_loss += loss.item()
            correct_samples += pred.eq(target).sum()

    avg_loss = total_loss / total_samples
    loss_history.append(avg_loss)

    print(f"\nAverage test loss: {avg_loss:.4f}  "
          f"Accuracy: {correct_samples}/{total_samples} "
          f"({100.0 * correct_samples / total_samples:.2f}%)\n")

In [ ]:
N_EPOCHS = 10

start_time = time.time()


train_loss_history, test_loss_history = [], []
for epoch in range(1, N_EPOCHS + 1):
    print('Epoch:', epoch)
    train_epoch(model, optimizer, train_loader, train_loss_history)
    evaluate(model, test_loader, test_loss_history)

print('Execution time:', '{:5.2f}'.format(time.time() - start_time), 'seconds')

In [ ]:
evaluate(model, test_loader, test_loss_history)

# Plot a confusion matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Confusion matrix
def plot_confusion_matrix(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = model(data)
            _, preds = torch.max(output, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(target.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

plot_confusion_matrix(model, test_loader)

# Visualize correct and wrong predictions

In [ ]:
# Visualize correct and wrong predictions
def plot_predictions(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_images = []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = model(data)
            _, preds = torch.max(output, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            all_images.extend(data.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_images = np.array(all_images)

    # Plot example images with predictions
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        ax.imshow(all_images[i][0], cmap='gray')
        ax.set_title(f'Label: {all_labels[i]}\nPred: {all_preds[i]}')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

plot_predictions(model, test_loader)

### Question:
Explain how the Transformer architecture can serve as a unified framework for both language and vision tasks?

*   Your answer


